# 밸런스팀 관점 — Isolation Forest 다변량 이상치 탐지 + SHAP

**원본 위치**: `이터널리턴_코드_정리.ipynb` cell 9~15 (섹션: `# Isolation Forest , SHAP 분석`)

**추정 담당**: 황현웅 (확실하지 않음, 팀원 확인 필요)
> 근거: KPT 회고에서 황현웅이 "2변량 분석의 한계를 인식한 뒤... 다변량 분석으로 확장했다"고 직접 서술했고, 이 서술이 여기 Isolation Forest+SHAP 코드의 목적과 정확히 일치합니다. 다만 이는 정황적 추정이며 코드에 작성자 표기는 없습니다.

**이 노트북은 원본 셀 내용을 그대로 보존합니다** (한글 폰트 설정이 이후 셀에서 주석 처리되어 있는 등, 실제로 있었던 순서 의존성도 그대로 남겨둠).
바로 실행 가능한 정리본이 필요하면 `../team_final/02_balance_and_newbie_analysis_consolidated.ipynb`를 사용하세요.

- 관련 문서: `../../docs/03_issues_and_troubleshooting.md` #5, #6, #11, #12
- 입력 파일: `EternalReturn_kakaogames_2024_character_added.csv` (`01_character_mapping_and_dashboard_prep.ipynb` 산출물)


## 설정 (한글 폰트, 상수)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# (선택) 한글 폰트 — Colab
!apt-get install -y fonts-nanum -q
import matplotlib.font_manager as fm
fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

CSV_PATH = "EternalReturn_kakaogames_2024_character_added.csv"  # 경로 맞추기
VERSION = 23           # 분석 대상 패치
CONTAMINATION = 0.10   # 이상치 비율(상위 10%) — 후보 개수 조절
EXCLUDE = ["알론소", "일레븐"]  # 역할 특성상 튀는 탱커 제외

## 캐릭터별 집계 (패치 23.0 기준, 6개 지표 + 픽률)

In [ ]:
df = pd.read_csv(CSV_PATH)
v = df[df["versionMajor"] == VERSION].copy()
total = len(v)

g = v.groupby("character_name_kr")
agg = pd.DataFrame({
    "표본수":       g.size(),
    "승률":         g["victory"].mean() * 100,
    "평균등수":     g["gameRank"].mean(),
    "평균생존시간": g["playTime"].mean(),
    "평균딜":       g["damageToPlayer"].mean(),
    "평균킬":       g["playerKill"].mean(),
})
agg["픽률"] = agg["표본수"] / total * 100
agg = agg.reset_index().rename(columns={"character_name_kr": "캐릭터"})
agg = agg[~agg["캐릭터"].isin(EXCLUDE)].reset_index(drop=True)
print(f"분석 캐릭터 {len(agg)}종 (v{VERSION})")
agg.head()

## Isolation Forest 이상치 탐지 (상위 10%)

In [ ]:
METRICS = ["승률", "픽률", "평균등수", "평균생존시간", "평균딜", "평균킬"]
Xs = StandardScaler().fit_transform(agg[METRICS])

iso = IsolationForest(n_estimators=400, contamination=CONTAMINATION, random_state=42)
iso.fit(Xs)
agg["비정상도"] = -iso.score_samples(Xs)
agg["이상치"] = (iso.predict(Xs) == -1)

wmean = agg["승률"].mean()
def direction(r):
    if not r["이상치"]:
        return "정상"
    return "너프 검토" if r["승률"] > wmean else "성능 개선 검토"
agg["점검방향"] = agg.apply(direction, axis=1)

후보 = agg[agg["이상치"]].sort_values("비정상도", ascending=False)
후보[["캐릭터", "승률", "픽률", "평균등수", "평균생존시간", "평균딜", "평균킬", "비정상도", "점검방향"]].round(2)

## 시각화: 픽률×승률 산점도 (이상치 라벨링)

→ 밸런스팀 슬라이드 13 원본 차트

In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))
wm, pm = agg["승률"].mean(), agg["픽률"].mean()
ax.axhline(wm, ls="--", c="#999", lw=0.9)
ax.axvline(pm, ls="--", c="#999", lw=0.9)

col = {"정상": "#C7CFDB", "너프 검토": "#F97362", "성능 개선 검토": "#5AA9E6"}
for cls, c in col.items():
    sub = agg[agg["점검방향"] == cls]
    ax.scatter(sub["픽률"], sub["승률"], s=(90 if cls != "정상" else 45),
               c=c, edgecolor="white", linewidth=0.8, label=cls, alpha=0.9)
for _, r in agg[agg["이상치"]].iterrows():
    ax.annotate(r["캐릭터"], (r["픽률"], r["승률"]), fontsize=10, fontweight="bold",
                xytext=(6, 4), textcoords="offset points")
ax.set_xlabel("픽률 (%)"); ax.set_ylabel("승률 (%)")
ax.set_title(f"다변량 이상치 탐지 — v{VERSION} 점검 후보", fontweight="bold")
ax.legend(loc="upper right")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("anomaly_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

## SHAP 분석 (1차: 전체 이상치 후보 대상 20,000 샘플)

⚠ `pip install shap`이 사전 설치되어 있어야 합니다 (원본 주석 그대로 유지 — 트러블슈팅 #12).

In [ ]:
# pip install shap 필요
import shap
from xgboost import XGBClassifier

feat = ["damageToPlayer", "playTime", "playerKill", "monsterKill", "craftRare", "craftEpic"]
sample = v.dropna(subset=feat + ["victory"]).sample(min(20000, len(v)), random_state=42)
X = sample[feat]
y = sample["victory"]

# 승리 여부(0/1) 분류 모델
model = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1,
                      subsample=0.8, eval_metric="logloss", random_state=42)
model.fit(X, y)

explainer = shap.TreeExplainer(model)
X_sample = X.sample(3000, random_state=42)
shap_values = explainer.shap_values(X_sample)
shap.summary_plot(shap_values, X_sample, feature_names=feat)

## SHAP 분석 (2차: 캐릭터 단위 재사용 버전)

`TARGET` 값만 바꾸면 다른 캐릭터에 그대로 재적용할 수 있도록 파라미터화된 버전. 위 셀과 달리 지원가 특성 지표(힐량/팀회복/보호막흡수/어시스트)를 포함해 더 풍부한 feature set을 씁니다.

⚠ 원본 코드에서 한글 폰트 설정 줄이 주석 처리되어 있습니다 — 위쪽 셀에서 이미 폰트가 등록된 상태를 전제하고 있어, 이 셀만 단독 실행하면 한글이 다시 깨질 수 있습니다 (트러블슈팅 #11).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
from xgboost import XGBClassifier

# (Colab 한글 폰트)
# !apt-get install -y fonts-nanum -q
# import matplotlib.font_manager as fm
# fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
# plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

CSV_PATH = "EternalReturn_kakaogames_2024_character_added.csv"
VERSION  = 23
TARGET   = "다르코"   # ← 분석할 캐릭터 이름만 바꾸면 됨

# 행동 지표 (gameRank 제외!). 지원가 특성 지표 포함.
FEATURES = {
    "damageToPlayer":  "딜",
    "playTime":        "생존시간",
    "playerKill":      "킬",
    "healAmount":      "힐량",
    "teamRecover":     "팀회복",
    "protectAbsorb":   "보호막흡수",
    "playerAssistant": "어시스트",
    "monsterKill":     "몬스터킬",
}

# ---- 데이터 로드 & 캐릭터 필터 ----
df = pd.read_csv(CSV_PATH)
v = df[df["versionMajor"] == VERSION]
ch = v[v["character_name_kr"] == TARGET].copy()

feat = list(FEATURES.keys())
s = ch.dropna(subset=feat + ["victory"])
X = s[feat]
y = s["victory"]
print(f"{TARGET}: 표본 {len(X)}건 · 승률 {y.mean()*100:.1f}%")

# ---- 승리 여부 예측 모델 ----
model = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1,
                      subsample=0.8, eval_metric="logloss", random_state=42)
model.fit(X, y)

# ---- SHAP ----
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X)

# 영향도 순위 + 방향(승률↑/↓)
mean_abs = np.abs(shap_values).mean(0)
print(f"\n{TARGET} 승률 기여 요인 (SHAP):")
for i in np.argsort(mean_abs)[::-1]:
    corr = np.corrcoef(X.iloc[:, i], shap_values[:, i])[0, 1]
    direction = "승률↑" if corr > 0 else "승률↓"
    print(f"  {FEATURES[feat[i]]:10s} {mean_abs[i]:.3f} ({direction})")

# ---- 시각화 (한글 라벨) ----
X_kr = X.rename(columns=FEATURES)
plt.figure()
shap.summary_plot(shap_values, X_kr, show=False)
plt.title(f"{TARGET} — 승률 기여 요인 (SHAP)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"shap_{TARGET}.png", dpi=140, bbox_inches="tight", facecolor="white")
plt.show()

# ============================================================
# 해석 가이드:
#   - 막대(점 분포)가 오른쪽으로 갈수록 그 지표가 승리에 기여.
#   - 색(빨강=값 높음)이 오른쪽에 모이면 "그 지표가 높을수록 이긴다".
#   - 딜이 '승률↓'로 나오면 → 딜 중심 플레이는 이 캐릭터에 안 맞는다는 뜻.
#   → 너프/버프 시 '승률을 만드는 핵심 지표'를 건드려야 효과가 있다.
#     (예: 샬럿이 어시스트·생존으로 이긴다면 딜 너프는 효과 없음)
# ============================================================


## 산출물 매핑
- 산점도 → 슬라이드 13
- SHAP 카드(나타폰/다르코/샬럿) → 슬라이드 14
- 결과표 8개 캐릭터 → 부록 슬라이드 22

## 코드에 없는 부분 (확인 필요)
- 슬라이드 15 "전체 vs 초보 유저 사분면 비교" 재현 코드 없음
- 슬라이드 12 "1차 승률×픽률 사분면 분류(패치 후 117,551건)" — 이 노트북의 cell 11~13과 동일 구조이나 표본수가 다름(패치23.0 전체 대비 117,551건). 버전/필터 차이가 있는 것으로 보이나 정확한 파라미터는 확인 불가.
